# Persistent retrieval artifact benchmark

Build portable exact and compact IVF-PQ artifacts for BioData ProtT5 layer 0. Compare approximate local retrieval, exact reranking, exhaustive pgvector, and full exact FAISS CPU and cuVS GPU search.

## tl;dr

This notebook is intentionally unexecuted. Its default `local` mode is a smoke test for the API and notebook workflow. Set `BIODATA_BENCHMARK_TARGET=nas` only when the remote HNSW build is not competing for database I/O. The final cells report recall@10, recall@50, recall@100, and RMSD against exhaustive pgvector results for 10 deterministic pseudo-random proteins.

## Context & Methods

### Key assumptions

- `embedding_type_id=3`, `layer_index=0` identifies the stored ProtT5 protein vectors.
- Cosine distance is the reference metric. IVF-PQ is trained with inner-product search after L2 normalization, so its raw approximate distance is `1 - score`.
- Recall@10, recall@50, and recall@100 compare each method's first K proteins with the exact pgvector first K. RMSD compares distances only for proteins shared with the exact top 100; reranked distances should therefore be exact apart from numeric representation.
- IVF-PQ builds in two bounded-memory PostgreSQL passes. The exact artifact is a separate float16 matrix plus protein-ID metadata; FAISS CPU and cuVS GPU materialize their exact states from that local artifact, not from PostgreSQL.
- The exhaustive pgvector baseline can take roughly 9 minutes per query on the remote database. Local-mode timings are integration checks, not production estimates.

## Setup

In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = next((path for path in (Path.cwd(), Path.cwd().parent) if (path / 'CBBIO').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the repository root or its notebooks directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from CBBIO import IndexBuildSpec, IndexKey, IndexManager, connect

In [ ]:
# Reproducible experiment parameters. Both config files are local-only and ignored by Git.
BENCHMARK_TARGET = os.environ.get('BIODATA_BENCHMARK_TARGET', 'nas').strip().lower()
if BENCHMARK_TARGET not in {'local', 'nas'}:
    raise ValueError("BIODATA_BENCHMARK_TARGET must be 'local' or 'nas'.")
CONFIG_PATH = PROJECT_ROOT / ('config_test.yaml' if BENCHMARK_TARGET == 'local' else 'config.yaml')
EXPECTED_HOST = 'localhost' if BENCHMARK_TARGET == 'local' else '192.168.49.224'
DATABASE_LABEL = f'biodata-{BENCHMARK_TARGET}'
EMBEDDING_TYPE_ID = 3
LAYER_INDEX = 0
METRIC = 'cosine'
QUERY_COUNT = 10
QUERY_SEED = 20260919991919
TOP_K = 100
RECALL_CUTOFFS = (10, 50, 100)
CANDIDATE_COUNT = 10_000
INDEX_ROOT = Path(os.environ.get('BIODATA_INDEX_ROOT', PROJECT_ROOT / '.biodata' / 'indexes'))

MAX_TRAINING_SAMPLE_SIZE = 124_239
SUBQUANTIZERS = 128
BITS_PER_CODE = 8
NPROBE = 256
HNSW_EF_SEARCH = 200
HNSW_CANDIDATE_POOL = 200
RUN_PGVECTOR_HNSW = BENCHMARK_TARGET == "nas"

if max(RECALL_CUTOFFS) > TOP_K:
    raise ValueError('TOP_K must be at least the largest recall cutoff.')

print(f'Benchmark target: {BENCHMARK_TARGET}; config: {CONFIG_PATH.name}; expected host: {EXPECTED_HOST}')
print(f'Index directory: {INDEX_ROOT}')
print(f'Queries: {QUERY_COUNT}; seed: {QUERY_SEED}; top-k: {TOP_K}; candidates: {CANDIDATE_COUNT}')

## Data

The candidate collection is `sequence_embeddings`, filtered to ProtT5 layer 0. Query proteins come from PostgreSQL `TABLESAMPLE SYSTEM ... REPEATABLE(seed)`, then a seeded hash ordering. This avoids a full-table `count(*)`, scan, or sort when choosing 100 reproducible random queries.

In [ ]:
client = connect(config_path=CONFIG_PATH)
if f'@{EXPECTED_HOST}:' not in client.dsn:
    raise RuntimeError(f'{CONFIG_PATH.name} must target {EXPECTED_HOST}, not a different database.')
server = client.query_one('SELECT current_database() AS database_name, inet_server_addr()::text AS server_address;')
print(f'Connected through {EXPECTED_HOST} to {server["database_name"]} (PostgreSQL address {server["server_address"]}).')


query_rows: list[dict[str, object]] = []
seen_proteins: set[str] = set()
for sample_attempt in range(20):
    if len(query_rows) == QUERY_COUNT:
        break
    table_sample_seed = ((QUERY_SEED + sample_attempt) % 999_983) / 999_983
    rows = client.query_all(
        f'''
        SELECT se.sequence_id, p.id AS protein_id
        FROM sequence_embeddings AS se TABLESAMPLE SYSTEM (0.1) REPEATABLE ({table_sample_seed})
        JOIN protein p ON p.sequence_id = se.sequence_id
        WHERE se.embedding_type_id = %s
          AND se.layer_index = %s
        ORDER BY hashtextextended(se.sequence_id::text, {QUERY_SEED + sample_attempt})
        LIMIT %s;
        ''',
        (EMBEDDING_TYPE_ID, LAYER_INDEX, QUERY_COUNT - len(query_rows)),
    )
    for row in rows:
        protein_id = str(row['protein_id'])
        if protein_id not in seen_proteins:
            query_rows.append({'protein_id': protein_id, 'sequence_id': int(row['sequence_id'])})
            seen_proteins.add(protein_id)

if len(query_rows) < QUERY_COUNT:
    raise RuntimeError('Could not obtain enough random query proteins after 20 deterministic samples.')

query_frame = pd.DataFrame(query_rows)
query_ids = query_frame['protein_id'].tolist()
query_sequence_ids = dict(zip(query_frame['protein_id'], query_frame['sequence_id'], strict=True))
display(query_frame)

## Build persistent retrieval artifacts

This cell first creates or reuses the portable exact float16 store. It is the shared local matrix
for FAISS CPU, cuVS GPU, and IVF-PQ. IVF-PQ is then derived from that store without another full
PostgreSQL vector transfer; both artifacts can be copied to a compute node.

In [ ]:
manager = IndexManager(INDEX_ROOT, search_nprobe=NPROBE)
key = IndexKey.from_biodata(
    client,
    database_label=DATABASE_LABEL,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
    metric=METRIC,
)
revision = client.embedding_index_revision(EMBEDDING_TYPE_ID, LAYER_INDEX)
vector_count = int(revision.split(';', maxsplit=1)[0].split('=', maxsplit=1)[1])
INDEX_SPEC = IndexBuildSpec(
    nlist=min(3476, max(1, int(np.sqrt(vector_count)))),
    subquantizers=SUBQUANTIZERS,
    bits_per_code=BITS_PER_CODE,
    training_sample_size=min(MAX_TRAINING_SAMPLE_SIZE, vector_count),
    nprobe=NPROBE,
)
print(f'Candidate vectors: {vector_count:,}; IVF-PQ nlist: {INDEX_SPEC.nlist:,}; search nprobe: {NPROBE:,}')

exact_inspection = manager.inspect_exact_store(
    database_label=DATABASE_LABEL,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
    source_revision=revision,
)
if exact_inspection.state == 'current':
    exact_build_seconds = 0.0
    exact_manifest = exact_inspection.manifest
    print('Reusing current portable exact store.')
elif exact_inspection.state == 'missing':
    started_at = time.perf_counter()
    exact_manifest = manager.build_exact_store(
        key,
        lambda: client.iter_protein_embedding_index_batches(
            EMBEDDING_TYPE_ID,
            LAYER_INDEX,
            batch_size=10_000,
        ),
        source_revision=revision,
    )
    exact_build_seconds = time.perf_counter() - started_at
    print(f'Built portable exact store with {exact_manifest.vector_count:,} vectors in {exact_build_seconds / 60:.1f} min.')
else:
    raise RuntimeError(
        f'Portable exact store is {exact_inspection.state}: {exact_inspection.reason}. '
        'Rebuild it deliberately with overwrite=True after deciding how to handle the old artifact.'
    )

if exact_manifest is not None:
    print('Portable exact-store build timings:')
    for label, seconds in (
        ('PostgreSQL vector read / transfer', exact_manifest.source_read_seconds),
        ('Save float16 matrix', exact_manifest.vector_write_seconds),
        ('Save SQLite protein-ID metadata', exact_manifest.metadata_write_seconds),
    ):
        if seconds is None:
            print(f'  {label}: unavailable (artifact predates timing instrumentation)')
        else:
            print(f'  {label}: {seconds:.3f} s')

inspection = manager.inspect(key, source_revision=revision)
if inspection.state == 'current':
    build_seconds = 0.0
    print('Reusing current persistent IVF-PQ index.')
elif inspection.state == 'missing':
    started_at = time.perf_counter()
    manifest = manager.build_ivf_pq(
        key,
        lambda: client.iter_protein_embedding_index_batches(
            EMBEDDING_TYPE_ID,
            LAYER_INDEX,
            batch_size=10_000,
        ),
        source_revision=revision,
        spec=INDEX_SPEC,
    )
    build_seconds = time.perf_counter() - started_at
    print(f'Built IVF-PQ from the local exact store with {manifest.vector_count:,} vectors in {build_seconds / 60:.1f} min.')
else:
    raise RuntimeError(
        f'Persistent index is {inspection.state}: {inspection.reason}. '
        'Rebuild it deliberately with overwrite=True after deciding how to handle the old artifact.'
    )

client.configure_persistent_index(manager, database_label=DATABASE_LABEL)
index = manager.load(key, source_revision=revision)
exact_store = manager.load_exact_store(
    database_label=DATABASE_LABEL,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
    source_revision=revision,
)

## Results

Run exhaustive pgvector first to establish ground truth. FAISS CPU and cuVS GPU perform full exact local searches from the portable exact artifact. The raw IVF-PQ path performs no PostgreSQL distance calculation. The persistent path retrieves the configured local candidate count and sends only those to PostgreSQL for exact reranking.

In [ ]:
# 2. Exhaustive local FAISS CPU search from the portable exact store.
started_at = time.perf_counter()
faiss_cpu_results = client.find_nearest_neighbors_for_proteins(
    query_ids,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
    k=TOP_K,
    metric=METRIC,
    include_query=False,
    backend='faiss_cpu',
    use_ann=False,
)
faiss_cpu_seconds = time.perf_counter() - started_at
print(f'Exact FAISS CPU: {faiss_cpu_seconds:.3f} s')

### Cold-start versus warm exact local search

The first exact call materializes the backend state from the local portable exact store (and, for cuVS, transfers/prepares it on the GPU). The repeated call uses that resident state. Both calls still fetch the query vectors and format their results, so `cold − warm` is an estimate of state loading/materialization rather than an isolated I/O measurement.

In [ ]:
# 2b. Repeat FAISS CPU with the exact-store-derived index state already resident.
started_at = time.perf_counter()
faiss_cpu_warm_results = client.find_nearest_neighbors_for_proteins(
    query_ids,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
    k=TOP_K,
    metric=METRIC,
    include_query=False,
    backend='faiss_cpu',
    use_ann=False,
)
faiss_cpu_warm_seconds = time.perf_counter() - started_at
faiss_cpu_load_estimate_seconds = max(0.0, faiss_cpu_seconds - faiss_cpu_warm_seconds)
print(f'FAISS CPU exact, warm: {faiss_cpu_warm_seconds:.3f} s')
print(f'FAISS CPU state loading/materialization (estimated): {faiss_cpu_load_estimate_seconds:.3f} s')

In [ ]:
# 3. Exhaustive local cuVS GPU search from the portable exact store into GPU memory.
started_at = time.perf_counter()
cuvs_gpu_results = client.find_nearest_neighbors_for_proteins(
    query_ids,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
    k=TOP_K,
    metric=METRIC,
    include_query=False,
    backend='cuvs_gpu',
    device='cuda:0',
    use_ann=False,
)
cuvs_gpu_seconds = time.perf_counter() - started_at
print(f'Exact cuVS GPU: {cuvs_gpu_seconds:.3f} s')

In [ ]:
# 3b. Repeat cuVS GPU with the exact-store-derived GPU search state already resident.
started_at = time.perf_counter()
cuvs_gpu_warm_results = client.find_nearest_neighbors_for_proteins(
    query_ids,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
    k=TOP_K,
    metric=METRIC,
    include_query=False,
    backend='cuvs_gpu',
    device='cuda:0',
    use_ann=False,
)
cuvs_gpu_warm_seconds = time.perf_counter() - started_at
cuvs_gpu_load_estimate_seconds = max(0.0, cuvs_gpu_seconds - cuvs_gpu_warm_seconds)
print(f'cuVS GPU exact, warm: {cuvs_gpu_warm_seconds:.3f} s')
print(f'cuVS GPU state loading/materialization (estimated): {cuvs_gpu_load_estimate_seconds:.3f} s')

### Verify where warm-call time is spent

Run this optional diagnostic if an exact-local timing is surprising. It first makes an unprofiled priming call for the selected backend, then profiles a second public API call that reuses that state. This distinguishes FAISS/cuVS engine time from Python, PostgreSQL query-vector loading, and result handling. It is diagnostic only and is excluded from `timing_frame`.

In [ ]:
# Optional: this primes the selected state, then profiles one additional warm exact batch.
PROFILE_EXACT_BACKEND = None

if PROFILE_EXACT_BACKEND is not None:
    import cProfile
    import io
    import pstats

    if PROFILE_EXACT_BACKEND not in {'faiss_cpu', 'cuvs_gpu'}:
        raise ValueError("PROFILE_EXACT_BACKEND must be 'faiss_cpu', 'cuvs_gpu', or None.")

    profile_kwargs = {
        'embedding_type_id': EMBEDDING_TYPE_ID,
        'layer_index': LAYER_INDEX,
        'k': TOP_K,
        'metric': METRIC,
        'include_query': False,
        'backend': PROFILE_EXACT_BACKEND,
        'use_ann': False,
    }
    if PROFILE_EXACT_BACKEND == 'cuvs_gpu':
        profile_kwargs['device'] = 'cuda:0'

    # The cache retains one state per compute resource, so explicitly prime the selected backend.
    _ = client.find_nearest_neighbors_for_proteins(query_ids, **profile_kwargs)

    profiler = cProfile.Profile()
    started_at = time.perf_counter()
    profiler.enable()
    _ = client.find_nearest_neighbors_for_proteins(query_ids, **profile_kwargs)
    profiler.disable()
    print(f'{PROFILE_EXACT_BACKEND} warm API call: {time.perf_counter() - started_at:.3f} s')

    profile_output = io.StringIO()
    pstats.Stats(profiler, stream=profile_output).strip_dirs().sort_stats('cumulative').print_stats(20)
    print(profile_output.getvalue())
else:
    print("Skipped. Set PROFILE_EXACT_BACKEND to 'faiss_cpu' or 'cuvs_gpu' to profile one warm batch.")

In [ ]:
# 1. Exhaustive pgvector ground truth (no HNSW or IVFFlat).
started_at = time.perf_counter()
exact_results = client.find_nearest_neighbors_for_proteins(
    query_ids,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
    k=TOP_K,
    metric=METRIC,
    include_query=False,
    backend='pgvector',
    use_ann=False,
)
exact_seconds = time.perf_counter() - started_at
print(f'Exact pgvector: {exact_seconds / 60:.1f} min')

### pgvector HNSW ANN

This comparison runs only in `nas` mode because the HNSW index is currently available on the remote reference database. It uses the configured `ef_search` and candidate pool. Returned distances are computed by pgvector for the retrieved neighbors, but recall remains approximate because HNSW may omit true neighbors.

In [ ]:
# 1b. Approximate pgvector HNSW retrieval on the remote reference database.
hnsw_results = None
hnsw_seconds = None

if RUN_PGVECTOR_HNSW:
    started_at = time.perf_counter()
    hnsw_results = client.find_nearest_neighbors_for_proteins(
        query_ids,
        embedding_type_id=EMBEDDING_TYPE_ID,
        layer_index=LAYER_INDEX,
        k=TOP_K,
        metric=METRIC,
        include_query=False,
        backend="pgvector",
        use_ann=True,
        ann_ef_search=HNSW_EF_SEARCH,
        ann_candidate_pool=HNSW_CANDIDATE_POOL,
    )
    hnsw_seconds = time.perf_counter() - started_at
    print(f"pgvector HNSW ANN: {hnsw_seconds:.3f} s (ef_search={HNSW_EF_SEARCH}, candidate pool={HNSW_CANDIDATE_POOL})")
else:
    print("Skipping pgvector HNSW ANN: this benchmark target has no configured remote HNSW index.")

In [ ]:
# 4. IVF-PQ only: scores are approximate cosine similarities, with no database reranking.
query_vectors = client.get_protein_embeddings(
    query_ids,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
)
started_at = time.perf_counter()
raw_candidates = {
    query_id: [
        candidate
        for candidate in manager.search(
            index,
            query_vectors[query_id],
            key=key,
            candidate_count=CANDIDATE_COUNT,
        )
        if candidate.sequence_id != query_sequence_ids[query_id]
    ][:TOP_K]
    for query_id in query_ids
}
raw_seconds = time.perf_counter() - started_at

candidate_sequence_ids = sorted({candidate.sequence_id for candidates in raw_candidates.values() for candidate in candidates})
candidate_rows = client.query_all(
    'SELECT sequence_id, id AS protein_id FROM protein WHERE sequence_id = ANY(%s);',
    (candidate_sequence_ids,),
)
protein_by_sequence_id = {int(row['sequence_id']): str(row['protein_id']) for row in candidate_rows}
raw_results = {
    query_id: [
        {
            'protein_id': protein_by_sequence_id[candidate.sequence_id],
            'distance': 1.0 - candidate.score,
        }
        for candidate in candidates
        if candidate.sequence_id in protein_by_sequence_id
    ]
    for query_id, candidates in raw_candidates.items()
}

print(f'Raw IVF-PQ: {raw_seconds:.3f} s (plus one ID-resolution query)')

In [ ]:
# 5. IVF-PQ candidates followed by exact pgvector reranking.
started_at = time.perf_counter()
reranked_results = client.find_nearest_neighbors_for_proteins(
    query_ids,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
    k=TOP_K,
    metric=METRIC,
    include_query=False,
    backend='faiss_persistent',
    ann_candidate_pool=CANDIDATE_COUNT,
)
reranked_seconds = time.perf_counter() - started_at
print(f'IVF-PQ + reranking: {reranked_seconds:.3f} s')

In [ ]:
def _neighbors_to_distances(neighbors):
    return {neighbor.protein_id: float(neighbor.distance) for neighbor in neighbors}

def _result_protein_ids(method_name, results):
    if method_name == 'IVF-PQ only':
        return [item['protein_id'] for item in results]
    return [neighbor.protein_id for neighbor in results]

def _evaluate_method(method_name, results):
    rows = []
    for query_id in query_ids:
        exact_protein_ids = [neighbor.protein_id for neighbor in exact_results[query_id]]
        method_protein_ids = _result_protein_ids(method_name, results[query_id])
        exact_by_protein = _neighbors_to_distances(exact_results[query_id])
        if method_name == 'IVF-PQ only':
            method_by_protein = {item['protein_id']: float(item['distance']) for item in results[query_id]}
        else:
            method_by_protein = _neighbors_to_distances(results[query_id])
        shared_proteins = sorted(set(exact_by_protein) & set(method_by_protein))
        squared_errors = [
            (method_by_protein[protein_id] - exact_by_protein[protein_id]) ** 2
            for protein_id in shared_proteins
        ]
        row = {
            'method': method_name,
            'query_id': query_id,
            'shared_neighbors': len(shared_proteins),
            'rmsd_distance': float(np.sqrt(np.mean(squared_errors))) if squared_errors else np.nan,
        }
        for cutoff in RECALL_CUTOFFS:
            exact_at_cutoff = set(exact_protein_ids[:cutoff])
            method_at_cutoff = set(method_protein_ids[:cutoff])
            row[f'recall_at_{cutoff}'] = len(exact_at_cutoff & method_at_cutoff) / cutoff
        rows.append(row)
    return rows

METHOD_COLORS = {
    "pgvector HNSW ANN": "#E15759",
    'pgvector exact': '#595959',
    'FAISS CPU exact': '#4C78A8',
    'cuVS GPU exact': '#F28E2B',
    'IVF-PQ only': '#B07AA1',
    'IVF-PQ + exact reranking': '#59A14F',
}
quality_rows = (
    _evaluate_method("pgvector exact", exact_results)
    + (_evaluate_method("pgvector HNSW ANN", hnsw_results) if hnsw_results is not None else [])
    + _evaluate_method("FAISS CPU exact", faiss_cpu_results)
    + _evaluate_method("cuVS GPU exact", cuvs_gpu_results)
    + _evaluate_method("IVF-PQ only", raw_results)
    + _evaluate_method("IVF-PQ + exact reranking", reranked_results)
)
quality_frame = pd.DataFrame(quality_rows)
timing_records = [
    {"method": "pgvector exact", "elapsed_seconds": exact_seconds},
    {"method": "FAISS CPU exact (cold)", "elapsed_seconds": faiss_cpu_seconds},
    {"method": "FAISS CPU exact (warm)", "elapsed_seconds": faiss_cpu_warm_seconds},
    {"method": "FAISS CPU state loading/materialization (estimated)", "elapsed_seconds": faiss_cpu_load_estimate_seconds},
    {"method": "cuVS GPU exact (cold)", "elapsed_seconds": cuvs_gpu_seconds},
    {"method": "cuVS GPU exact (warm)", "elapsed_seconds": cuvs_gpu_warm_seconds},
    {"method": "cuVS GPU state loading/materialization (estimated)", "elapsed_seconds": cuvs_gpu_load_estimate_seconds},
    {"method": "IVF-PQ only", "elapsed_seconds": raw_seconds},
    {"method": "IVF-PQ + exact reranking", "elapsed_seconds": reranked_seconds},
]
if hnsw_seconds is not None:
    timing_records.insert(1, {"method": f"pgvector HNSW ANN (ef_search={HNSW_EF_SEARCH})", "elapsed_seconds": hnsw_seconds})
timing_frame = pd.DataFrame(timing_records)
summary_frame = quality_frame.groupby('method', as_index=False).agg(
    mean_recall_at_10=('recall_at_10', 'mean'),
    mean_recall_at_50=('recall_at_50', 'mean'),
    mean_recall_at_100=('recall_at_100', 'mean'),
    min_recall_at_100=('recall_at_100', 'min'),
    mean_rmsd_distance=('rmsd_distance', 'mean'),
)
display(timing_frame)
display(summary_frame)
display(quality_frame.sort_values(['method', 'recall_at_100', 'query_id']))

### Inspect exact disagreements

Strict recall can differ when a near-tie crosses a cutoff. This diagnostic covers exhaustive local methods and IVF-PQ plus exact reranking. It expands pgvector only for disagreeing queries and reports the missing and substituted proteins, their pgvector ranks, and their distance from each cutoff. A reranked missing protein outside the tie tolerance was absent from the IVF-PQ candidate pool.

In [ ]:
# Diagnose whether exhaustive local results differ only at an ambiguous pgvector cutoff.
DIAGNOSTIC_MARGIN = 20
TIE_TOLERANCE = 1e-6

exact_method_results = {
    'FAISS CPU exact': faiss_cpu_results,
    'cuVS GPU exact': cuvs_gpu_results,
    'IVF-PQ + exact reranking': reranked_results,
}
disagreement_queries = sorted(
    {
        query_id
        for results in exact_method_results.values()
        for query_id in query_ids
        if any(
            {neighbor.protein_id for neighbor in exact_results[query_id][:cutoff]}
            != {neighbor.protein_id for neighbor in results[query_id][:cutoff]}
            for cutoff in RECALL_CUTOFFS
        )
    }
)

expanded_exact_results = client.find_nearest_neighbors_for_proteins(
    disagreement_queries,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
    k=TOP_K + DIAGNOSTIC_MARGIN,
    metric=METRIC,
    include_query=False,
    backend='pgvector',
    use_ann=False,
)

diagnostic_rows = []
for method_name, results in exact_method_results.items():
    for query_id in disagreement_queries:
        expanded = expanded_exact_results[query_id]
        expanded_rank = {neighbor.protein_id: rank for rank, neighbor in enumerate(expanded, start=1)}
        expanded_distance = {neighbor.protein_id: neighbor.distance for neighbor in expanded}
        for cutoff in RECALL_CUTOFFS:
            reference = exact_results[query_id][:cutoff]
            observed = results[query_id][:cutoff]
            reference_ids = {neighbor.protein_id for neighbor in reference}
            observed_ids = {neighbor.protein_id for neighbor in observed}
            if reference_ids == observed_ids:
                continue
            cutoff_distance = reference[-1].distance
            for kind, protein_ids in (
                ('missing_from_local', sorted(reference_ids - observed_ids)),
                ('extra_in_local', sorted(observed_ids - reference_ids)),
            ):
                for protein_id in protein_ids:
                    pgvector_distance = expanded_distance.get(protein_id)
                    diagnostic_rows.append(
                        {
                            'method': method_name,
                            'query_id': query_id,
                            'cutoff': cutoff,
                            'difference': kind,
                            'protein_id': protein_id,
                            'pgvector_rank': expanded_rank.get(protein_id),
                            'pgvector_distance': pgvector_distance,
                            'distance_from_cutoff': (
                                None if pgvector_distance is None else pgvector_distance - cutoff_distance
                            ),
                            'cutoff_distance': cutoff_distance,
                            'within_tie_tolerance': (
                                None
                                if pgvector_distance is None
                                else abs(pgvector_distance - cutoff_distance) <= TIE_TOLERANCE
                            ),
                        }
                    )

exact_disagreement_frame = pd.DataFrame(diagnostic_rows)
if exact_disagreement_frame.empty:
    print('No strict top-100 disagreements between pgvector and the local exhaustive methods.')
else:
    display(exact_disagreement_frame.sort_values(['method', 'query_id', 'cutoff', 'difference', 'pgvector_rank']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
recall_columns = ['mean_recall_at_10', 'mean_recall_at_50', 'mean_recall_at_100']
recall_labels = ['recall@10', 'recall@50', 'recall@100']
positions = np.arange(len(recall_columns))
bar_width = 0.16
for method_index, (_, method_row) in enumerate(summary_frame.iterrows()):
    axes[0].bar(
        positions + (method_index - (len(summary_frame) - 1) / 2) * bar_width,
        [method_row[column] for column in recall_columns],
        width=bar_width,
        color=METHOD_COLORS[method_row['method']],
        label=method_row['method'],
    )
axes[0].set_xticks(positions, recall_labels)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Mean recall')
axes[0].set_title('Recovery against exact pgvector')
axes[0].legend()

axes[1].bar(
    summary_frame['method'],
    summary_frame['mean_rmsd_distance'],
    color=[METHOD_COLORS[method] for method in summary_frame['method']],
)
axes[1].set_ylabel('Mean RMSD of shared distances')
axes[1].set_title('Distance fidelity')
axes[1].tick_params(axis='x', rotation=15)

fig.tight_layout()
plt.show()

### Per-query recall

Each point is one query protein. Use the sorted per-query table above to identify its sequence ID.

In [ ]:
method_names = summary_frame['method'].tolist()
recall_distributions = [
    quality_frame.loc[quality_frame['method'] == method_name, 'recall_at_100'].to_numpy()
    for method_name in method_names
]
palette = [METHOD_COLORS[method_name] for method_name in method_names]

fig, axis = plt.subplots(figsize=(11, 4))
boxplot = axis.boxplot(
    recall_distributions,
    tick_labels=method_names,
    patch_artist=True,
    showmeans=True,
    medianprops={'color': '#222222'},
    meanprops={'marker': 'D', 'markerfacecolor': '#222222', 'markeredgecolor': '#222222'},
)
for patch, color in zip(boxplot['boxes'], palette, strict=True):
    patch.set_facecolor(color)
    patch.set_alpha(0.35)
for position, values in enumerate(recall_distributions, start=1):
    offsets = np.linspace(-0.08, 0.08, num=len(values))
    axis.scatter(position + offsets, values, color='#222222', s=28, zorder=3)

axis.set_ylim(0, 1.02)
axis.set_ylabel('recall@100 per query protein')
axis.set_title('Tail recall across query proteins')
axis.tick_params(axis='x', rotation=25)
axis.grid(axis='y', color='#DDDDDD', linewidth=0.8)
fig.tight_layout()
plt.show()

## Takeaways

Interpret the executed table as follows:

- FAISS CPU and cuVS GPU should match the pgvector reference apart from numeric representation. Their first timings materialize backend state from the portable exact store.
- IVF-PQ-only recall@10, recall@50, and recall@100 measure retrieval quality before PostgreSQL. Its distance RMSD reflects PQ approximation error.
- IVF-PQ plus reranking should have near-zero RMSD for recovered neighbors because PostgreSQL recomputes their exact cosine distances. Recall remains bounded by the local candidate pool.
- Increase `CANDIDATE_COUNT` or `nprobe` if reranked recall@K is insufficient; record the added latency and database load with the result.
- pgvector HNSW ANN measures the remote index with the configured `ef_search`. Its recovered distances are pgvector cosine distances, but recall is limited by approximate retrieval; increase `HNSW_EF_SEARCH` and record its latency and recall tradeoff.